# Modal: cloud GPUs from your notebook

Your M5 is a good machine for learning, and module 07 fine-tuned a 1.5B model on it in a few minutes. It runs out of room quickly, though: a 7B model in bf16, a 32B model, or anything that needs more than 24 GB of memory won't fit, and a bigger GPU would finish your small jobs faster too. Modal lets you rent those machines by the second, straight from Python. You write a normal function, say what hardware it needs, and call it. Modal runs it in the cloud and sends back the result.

**In this notebook**

1. What Modal is, and the five words you need
2. Check your setup, and read the live prices
3. Your first function
4. Images: the software your function needs
5. A GPU function, compared with your M5
6. Running many things at once with `map`
7. Volumes and secrets
8. A model that stays loaded
9. From notebook to a deployed endpoint
10. What it costs, and how to stay in control

**Before you start**

- Prerequisites: module 02 (the GPU timing section) helps, and so do modules 04 and 08. No apps need to be running.
- Your Modal login is already set up on this Mac, and the next cells check it. On a new machine, run `uv run modal setup` once. It opens a browser to log you in and saves a token in `~/.modal.toml`, in your home folder. Like your Hugging Face and W&B logins, that token stays outside `ML_Lab`.
- This notebook spends real money, a little at a time. Modal bills per second of compute, and the cells that cost something say so right above them. With the default switches the whole notebook should come to well under a dollar, and it prints its running estimates. Before you run anything, open your Modal dashboard and set a spending limit on your workspace, so a mistake can't turn into a surprise.
- Two switches in the setup cell control the spending. `RUN_CLOUD = False` lets you read and run everything without touching Modal. `RUN_BIGGER_GPUS` stays `False` until you decide to try A100 and H100 machines in the exercises.
- Modal's images and your data live in Modal's cloud, so nothing from this module lands in `ML_Lab/.cache`.
- Time: about 45 minutes. The first run of the GPU sections waits a few minutes while Modal builds an image with PyTorch in it. Later runs reuse it.

In [ ]:
import json
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import modal

# True runs the cloud cells for real. Each one says what it costs. False skips them all, for free.
RUN_CLOUD = True
# The exercises that use A100 and H100 machines stay off until you switch this on (a few cents each).
RUN_BIGGER_GPUS = False

# Notebooks run with their own folder as the working directory.
assert Path.cwd().name == "09_modal", f"Run this notebook from lessons/09_modal (working directory: {Path.cwd()})"
os.environ["UV_NO_PROGRESS"] = "1"
# Notebook kernels set FORCE_COLOR, which makes Modal draw animated spinners that clutter the output. Turn it off.
for var in ("FORCE_COLOR", "CLICOLOR_FORCE"):
    os.environ.pop(var, None)

print("modal", modal.__version__, "| local Python", sys.version.split()[0], "| RUN_CLOUD =", RUN_CLOUD)


def on_modal(app, call):
    """Run `call()` inside an ephemeral Modal app, or skip it when RUN_CLOUD is False.

    Section 3 shows what this does step by step. From section 5 on, it keeps the cells short.
    """
    if not RUN_CLOUD:
        print("RUN_CLOUD is False, so this cell skipped the cloud run (no cost).")
        return None
    def run():
        with modal.enable_output(), app.run():
            return call()

    # A separate thread, because Jupyter already has an event loop running and Modal's `.map()` refuses to
    # iterate inside a running loop. Plain `.remote()` calls work either way.
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(run).result()

A line that starts with `!` runs as a shell command, as in module 00. `modal` is the same package you imported above, and its command-line tool is installed in `.venv`, so `uv run modal ...` works from here and from iTerm2.

## 1. What Modal is, and the five words you need

Modal is serverless compute for Python. "Serverless" doesn't mean there are no servers. It means you never set one up, patch one, or leave one running. The loop looks like this:

```
your notebook                         Modal's cloud
-------------                         -------------------------------------------
fn.remote(args)  ---- args ---->      1. builds an image (your packages), cached after the first time
                                      2. starts a container from it on a machine with the GPU you asked for
                 <--- result ----     3. runs your function there, and returns what it returned
                                      4. keeps the container warm for a while, then shuts it down
```

You're billed by the second for the time containers run, so an idle app costs nothing. Scaling works the same way in the other direction: ask for 100 calls at once and Modal starts as many containers as it needs.

Five words cover most of what you'll write:

| Word | What it is |
|---|---|
| App | A named group of functions that deploy and stop together. `modal.App("name")` |
| Image | The software in the container: Python, packages, system tools. `modal.Image.debian_slim().uv_pip_install("torch")` |
| Function | A Python function that runs in a container. `@app.function(gpu="T4", image=...)` |
| `.remote()` | Call a function in the cloud and wait for the answer. `.map()` does the same for many inputs in parallel |
| Volume and Secret | Storage that outlives containers (model weights), and credentials passed in as environment variables |

Where Modal fits next to what you've used:

- Your M5 is free per run and has no setup, but it has 24 GB and one modest GPU. Use it to learn and to iterate.
- A hosted API (module 08's cost section) means no machine at all, and you pay per token. You can't change the model.
- A rented GPU box (RunPod, Lambda, Vast.ai) is cheapest per hour for long, steady jobs, and you manage it yourself. You pay while it's on, whether or not you use it.
- Modal is the middle path: your own code and models, no machine to manage, and per-second billing that scales to zero. The price is cold starts (the wait while a new container starts) and a higher hourly rate than a box you keep busy.

For a product, the trade is variable cost against fixed cost. Traffic that comes in bursts, jobs you run a few times a week, and prototypes all favor Modal. A GPU that would sit busy 24 hours a day usually favors a reserved machine. Section 10 turns that into numbers.

## 2. Check your setup, and read the live prices

Two quick checks. First, which Modal account this Mac is logged into. Then Modal's current prices, which the command-line tool can print, so you never need to trust a number from a tutorial (including this one).

In [ ]:
!uv run modal profile current

In [ ]:
# Reading prices is free. The fallback keeps the notebook usable offline (a snapshot, so don't trust it for decisions).
FALLBACK_PER_HOUR = {"T4": 0.59, "L4": 0.80, "A10": 1.10, "L40S": 1.95, "A100-40GB": 2.10, "A100-80GB": 2.50, "H100": 3.95}
KEY = {"T4": "t4", "L4": "l4", "A10": "a10g", "L40S": "l40s", "A100-40GB": "a100_40gb", "A100-80GB": "a100_80gb", "H100": "h100"}
try:
    # Notebook kernels set FORCE_COLOR, which makes the CLI wrap its JSON in color codes. Turn that off for the child.
    plain = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")} | {"NO_COLOR": "1"}
    rates = json.loads(subprocess.run([sys.executable, "-m", "modal", "billing", "rates", "--json"],
                                      capture_output=True, text=True, check=True, timeout=60, env=plain).stdout)
    PRICE_PER_HOUR = {gpu: float(rates[f"gpu_hour_cost_{key}"]) for gpu, key in KEY.items()}
    CPU_CORE_HOUR = float(rates["cpu_hour_cost"])
    MEM_GIB_HOUR = float(rates["mem_gib_hour_cost"])
    VOLUME_GIB_MONTH = float(rates["volume_storage_gib_month_cost"])
    source = "live from Modal"
except Exception as err:
    PRICE_PER_HOUR, CPU_CORE_HOUR, MEM_GIB_HOUR, VOLUME_GIB_MONTH = FALLBACK_PER_HOUR, 0.0473, 0.008, 0.09
    source = f"fallback snapshot ({type(err).__name__})"


def gpu_dollars(gpu: str, seconds: float) -> float:
    """What `seconds` of one GPU costs. Billing is per second, so this is just rate / 3600 * seconds."""
    return PRICE_PER_HOUR[gpu] / 3600 * seconds


print(f"Prices ({source}), dollars per hour:")
for gpu, price in PRICE_PER_HOUR.items():
    print(f"  {gpu:10} ${price:5.2f}/h   = ${price / 3600:.6f} per second")
print(f"  CPU        ${CPU_CORE_HOUR}/core/h, memory ${MEM_GIB_HOUR}/GiB/h, Volume storage ${VOLUME_GIB_MONTH}/GiB/month")

What to notice: a T4 costs a fraction of a cent for every few seconds you use it. The expensive part of cloud GPUs is never a short experiment. It's forgetting something that keeps running, which is why every cell below has a timeout, and section 10 ends with a cleanup checklist. (`A10` in the table is Modal's name for the A10G.)

## 3. Your first function

Start on CPU, where mistakes cost almost nothing. The function below reports where it runs. You'll run it twice: here on your Mac with `.local()`, and in Modal's cloud with `.remote()`.

Two details specific to notebooks. A Function that you define in a notebook is sent to the cloud as a pickled object, so Modal needs the same Python version there as here (the default image matches your local version, 3.14). And you start it inside `app.run()`, which creates a temporary app that exists only until the `with` block ends. Wrapping that in `modal.enable_output()` shows Modal's progress and logs.

Cost: a fraction of a cent. The first call also builds the base image, which takes a few seconds.

In [ ]:
app_intro = modal.App("ml-lab-intro")


@app_intro.function()
def where_am_i(name: str) -> dict:
    import os
    import platform
    import socket
    import sys

    return {
        "greeting": f"hello, {name}",
        "os": platform.platform(),
        "python": sys.version.split()[0],
        "cpu_cores": os.cpu_count(),
        "host": socket.gethostname(),
    }


print("on your Mac:", where_am_i.local("Pujan"))

In [ ]:
# The long form, once. Everything inside the `with` blocks runs in Modal's cloud and bills by the second.
if RUN_CLOUD:
    with modal.enable_output():
        with app_intro.run():
            print("in Modal  :", where_am_i.remote("Pujan"))
else:
    print("RUN_CLOUD is False: skipped.")

What to notice:

- The cloud answer says Linux, not macOS, and the kernel and host name are Modal's sandbox, not yours. Your function ran in a container on someone else's machine, and the `return` value came back to your notebook as an ordinary Python dict.
- The Python minor version matches your local one (3.14), because the default image copies it. The patch number can differ. If the minor versions ever differed, notebook-defined functions would fail to unpickle in the cloud.
- The `with` block ended, so the app stopped. Nothing is left running. This is an ephemeral app. Section 9 covers deployed apps, which stay up.
- Calling `.remote()` again in a new `app.run()` is faster than the first call, because the image is already built and cached.
- If you see `RESOURCE_EXHAUSTED ... Retrying` in the output, Modal is rate-limiting how quickly you create apps. The client retries on its own, so just wait.

From here on, `on_modal(app, lambda: ...)` from the setup cell wraps exactly this `with` pattern, so the cells stay short.

## 4. Images: the software your function needs

The container starts from an image, a snapshot of an operating system plus your packages. The default is a bare Python. You describe anything extra in code, and Modal builds it once and caches it, keyed by exactly what you described. Change one package and only that layer rebuilds.

```python
modal.Image.debian_slim()                                   # Python on a small Debian Linux
    .uv_pip_install("torch", "transformers")                # packages, installed with uv (like module 00)
    .apt_install("ffmpeg")                                  # system programs
    .env({"HF_HOME": "/models/hf"})                         # environment variables
    .add_local_file("config.json", "/root/config.json")     # files from your Mac
```

You'll use three images in this notebook. Defining them runs nothing, so these cells are free:

In [ ]:
# Python matches your local version by default, so notebook-defined functions can be unpickled in the cloud.
torch_image = modal.Image.debian_slim().uv_pip_install("torch", "numpy")      # section 5
hf_image = modal.Image.debian_slim().uv_pip_install("huggingface_hub")        # section 7
llm_image = modal.Image.debian_slim().uv_pip_install("torch", "transformers", "accelerate")  # section 8
print("three images defined; none has been built yet")

An image is built the first time an app that uses it runs, in Modal's cloud, not on your Mac. The torch image is large, so expect a few minutes the first time. After that it's cached, and the same image starts in seconds. The build runs on CPU, never on a GPU, and the GPU is the expensive part. (Check Modal's pricing page for how builds are billed.)

Libraries go inside the function, as in section 3 (`import torch` appears inside the function body). Your Mac's Python doesn't need them. In this module's case it already has torch, but your notebook's kernel and the cloud container are separate environments, and that separation is the whole point.

## 5. A GPU function, compared with your M5

Asking for a GPU is one argument: `gpu="T4"`. The function below is the matrix-multiply benchmark from module 02, section 5, with a GPU on the other end. It also records which GPU it got, using `nvidia-smi`, the tool that ships with NVIDIA drivers.

Cost: a T4 is billed at the rate printed in section 2, only while the container runs. This section is a minute or two of T4, plus the first image build. That comes to a few cents at most.

In [ ]:
app_gpu = modal.App("ml-lab-gpu", image=torch_image)


def matmul_benchmark_impl(n: int = 4096, dtype_name: str = "float16", reps: int = 20) -> dict:
    """The module 02 benchmark: time n x n matrix multiplies on whatever GPU this container has."""
    import subprocess
    import time

    import torch

    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    dtype = getattr(torch, dtype_name)
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    a @ b
    torch.cuda.synchronize()          # GPU work is asynchronous: sync before and after timing
    start = time.perf_counter()
    for _ in range(reps):
        a @ b
    torch.cuda.synchronize()
    seconds = (time.perf_counter() - start) / reps
    return {"gpu": gpu, "dtype": dtype_name, "n": n, "ms": round(seconds * 1000, 2),
            "tflops": round(2 * n**3 / seconds / 1e12, 1)}


# The decorator is just a function call, so you can apply it to a plain function you already wrote.
matmul_t4 = app_gpu.function(gpu="T4", timeout=300, name="matmul_t4")(matmul_benchmark_impl)

In [ ]:
import torch


def local_matmul(n: int = 4096, dtype=torch.float16, reps: int = 20) -> dict:
    """The same benchmark on your Mac's GPU (MPS), for a side-by-side comparison."""
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    sync = torch.mps.synchronize if device.type == "mps" else (lambda: None)
    a @ b
    sync()
    start = time.perf_counter()
    for _ in range(reps):
        a @ b
    sync()
    seconds = (time.perf_counter() - start) / reps
    return {"gpu": f"Apple {device.type}", "dtype": str(dtype).split(".")[-1], "n": n,
            "ms": round(seconds * 1000, 2), "tflops": round(2 * n**3 / seconds / 1e12, 1)}


print("your Mac:", local_matmul())

In [ ]:
t0 = time.perf_counter()
result = on_modal(app_gpu, lambda: (matmul_t4.remote(4096, "float16"), matmul_t4.remote(4096, "float32")))
wall = time.perf_counter() - t0
if result:
    for r in result:
        print("Modal:", r)
    print(f"\nWall clock for the whole cell: {wall:.0f} s (includes the image build on the first run and the container start).")
    print(f"The T4 itself was only busy for a few of those seconds, and you pay for container time, about ${gpu_dollars('T4', wall):.3f} at most.")

What to notice:

- `tflops` is trillions of floating-point operations per second, computed as $2n^3 / \text{time}$. A matrix multiply of two $n \times n$ matrices does $n^3$ multiplies and about $n^3$ adds, so $2n^3$ operations. That lets you compare machines with one number.
- Compare float16 and float32 on the T4. GPUs have dedicated hardware for half precision, so float16 is usually several times faster, which is part of why LLMs run in 16-bit or lower (module 01's memory table).
- Compare the T4 with your Mac. In a test run the T4 managed about 24 TFLOPS in float16 against about 15 for the M5, so a cheap data-center GPU is only modestly faster than your laptop at this size. The point of a cloud GPU is the range: the same code runs on an L4, an A10 or an H100 by changing one string (exercises 1 and 3), and those are several times faster.
- The wall-clock number is much bigger than the milliseconds the matmuls took. Most of it was starting the container, and on the first run, building the image. This overhead is why Modal suits jobs that run for a while, and why section 8 keeps a model loaded between calls.

## 6. Running many things at once with `map`

`.map()` takes many inputs and runs them in parallel, starting as many containers as it needs. You write the single-item function, and Modal does the fan-out. This is how you'd run a hyperparameter sweep, score a dataset, or embed a million documents.

The demo uses a function that just sleeps for 2 seconds, so you can see parallelism without a lot of compute. Run one by one, 12 of them would take 24 seconds.

Cost: CPU only, 12 short runs, far under a cent.

In [ ]:
app_map = modal.App("ml-lab-map")


@app_map.function()
def slow_square(x: int) -> int:
    import time

    time.sleep(2)          # stands in for real work
    return x * x


def run_map():
    start = time.perf_counter()
    results = list(slow_square.map(range(12)))
    return results, time.perf_counter() - start


out = on_modal(app_map, run_map)
if out:
    results, seconds = out
    print(results)
    print(f"12 inputs x 2 s each = 24 s one by one. With map it took {seconds:.1f} s (container startup included).")

What to notice: the loop is the same size as the sequential one, but the time was about 7 seconds in a test run instead of 24, because many containers ran at once. You pay for the total container time either way, so parallelism is roughly free in money and big in time. The catch is that every container pays its own startup, so tiny tasks (milliseconds each) should be batched into bigger chunks before you `map` them.

By default Modal will scale up to a large number of containers. On a GPU function that could get expensive fast, so for those, set `max_containers=` (for example `max_containers=4`) to put a ceiling on it.

## 7. Volumes and secrets

Containers are disposable: when one shuts down, everything it wrote disappears. Two things you'll need almost immediately don't fit that.

A Volume is a named disk that outlives containers. Its main use is model weights. Downloading a 15 GB model on every cold start would be slow and wasteful, so you download it once into a Volume and every container mounts it. A Secret is a set of environment variables injected into the container, so API keys never appear in your code or your notebook.

This cell downloads Qwen2.5-0.5B-Instruct (the model from module 04, about 1 GB) into a Volume called `ml-lab-models`, and it happens in Modal's data center, on their network, not through your Mac. Cost: a minute or less of CPU, plus storing 1 GB in the Volume at the price printed in section 2, about 9 cents per GiB per month. Section 10 shows how to delete the Volume.

In [ ]:
vol = modal.Volume.from_name("ml-lab-models", create_if_missing=True)
MODEL_DIR = "/models"
QWEN = "Qwen/Qwen2.5-0.5B-Instruct"

app_store = modal.App("ml-lab-storage", image=hf_image)


@app_store.function(volumes={MODEL_DIR: vol}, timeout=900)
def fetch_model(repo_id: str) -> str:
    import time
    from pathlib import Path

    from huggingface_hub import snapshot_download

    target = Path(MODEL_DIR) / repo_id.replace("/", "--")
    if (target / "config.json").exists():
        return f"already in the Volume, nothing to download: {target}"
    start = time.perf_counter()
    snapshot_download(repo_id, local_dir=target, allow_patterns=["*.json", "*.safetensors", "*.txt"])
    vol.commit()                       # make the new files visible to other containers
    size = sum(f.stat().st_size for f in target.rglob("*") if f.is_file())
    return f"downloaded {size / 1e9:.2f} GB in {time.perf_counter() - start:.0f} s into {target}"


print(on_modal(app_store, lambda: fetch_model.remote(QWEN)))

In [ ]:
# Run it again. The files are in the Volume now, so the second call does no download.
print(on_modal(app_store, lambda: fetch_model.remote(QWEN)))

Secrets work the same way, with one command to create them. For a gated model like Llama you'd store your Hugging Face token once, in iTerm2 (the token is yours to paste, not something to put in a notebook):

```bash
uv run modal secret create huggingface HF_TOKEN=hf_your_token_here
```

and attach it with `@app.function(secrets=[modal.Secret.from_name("huggingface")])`. Inside the function, `os.environ["HF_TOKEN"]` is set, and `huggingface_hub` picks it up on its own. The same pattern covers a W&B key (`WANDB_API_KEY`). This demo shows the mechanism without a real credential, using a Secret built from a dict:

In [ ]:
app_secret = modal.App("ml-lab-secret")


@app_secret.function(secrets=[modal.Secret.from_dict({"COURSE": "ml-lab", "NOT_A_REAL_KEY": "demo-value"})])
def read_secret() -> dict:
    import os

    return {"COURSE": os.environ.get("COURSE"), "NOT_A_REAL_KEY": os.environ.get("NOT_A_REAL_KEY")}


print(on_modal(app_secret, lambda: read_secret.remote()))

What to notice:

- The Volume is a normal folder inside the container (`/models`). Your code uses ordinary file paths, with no special API, apart from `vol.commit()` after writing so other containers can see the files. A container that starts later sees committed files automatically. One that was already running calls `vol.reload()` to see new ones.
- The second `fetch_model` call returned immediately. That's the whole point of a Volume.
- The Secret's values appeared as environment variables. In real use you'd create them once with `modal secret create`, and your code would name them with `modal.Secret.from_name(...)`, so nothing sensitive lives in the notebook or in git.
- `uv run modal volume ls ml-lab-models` and `uv run modal volume list` show what's stored from your terminal.

## 8. A model that stays loaded

Loading a model takes seconds, and running it takes milliseconds. If every call reloaded the model, you'd mostly pay to wait. The fix is a Modal class: `@modal.enter()` runs once when a container starts (load the model here), and each `@modal.method()` reuses the loaded model. After the last call, the container stays warm for `scaledown_window` seconds, then shuts down. While it's warm, calls are fast. After it shuts down, the next call is a cold start again.

This is the shape of every model server you'll build on Modal, and it's the cloud version of what `mlx_lm.server` did in module 08. The class below serves Qwen2.5-0.5B-Instruct on a T4, reading the weights from the Volume you filled in section 7.

Cost: the container is on for the first call's cold start plus the second call plus the `scaledown_window` of 120 seconds, so a few minutes of T4, about a cent or two. The first run also builds the image, on CPU.

In [ ]:
app_llm = modal.App("ml-lab-llm", image=llm_image)


@app_llm.cls(gpu="T4", volumes={MODEL_DIR: vol}, scaledown_window=120, timeout=600, serialized=True)
class QwenServer:
    @modal.enter()
    def load(self):
        import time

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        path = f"{MODEL_DIR}/{QWEN.replace('/', '--')}"
        start = time.perf_counter()
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16).to("cuda")
        print(f"model loaded in {time.perf_counter() - start:.1f} s")        # runs once per container

    @modal.method()
    def generate(self, prompt: str, max_new_tokens: int = 60) -> str:
        messages = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                                    return_tensors="pt", return_dict=True).to("cuda")
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def ask_twice():
    server = QwenServer()
    t0 = time.perf_counter()
    first = server.generate.remote("In one sentence, what is a GPU?")
    t1 = time.perf_counter()
    second = server.generate.remote("In one sentence, what is a neural network?")
    t2 = time.perf_counter()
    return first, t1 - t0, second, t2 - t1


out = on_modal(app_llm, ask_twice)
if out:
    first, cold, second, warm = out
    print(f"call 1 ({cold:5.1f} s, cold start + model load): {first.strip()}")
    print(f"call 2 ({warm:5.1f} s, container already warm): {second.strip()}")

What to notice:

- Call 1 includes starting a container and loading the model. Call 2 is only the generation. In a test run it was about 16 seconds for call 1 and under a second for call 2. Your own numbers will differ, but the gap is the lesson. A "serverless" endpoint that scales to zero pays that cold start whenever traffic resumes after a quiet period.
- Three dials control the trade-off. `scaledown_window` is how long an idle container stays warm (longer means fewer cold starts and more idle billing). `min_containers=1` keeps one container warm all the time, so users never hit a cold start, and you pay for the GPU around the clock. `max_containers` is the ceiling on scale-out. Section 10 puts dollar figures on the first two.
- The model sits in the Volume, so even a cold start only has to read it from disk, not download it. A larger model makes both loading and the cold start take longer, and Modal has options (memory snapshots, faster loaders) for squeezing that in production.
- Compare this to the fine-tuned router from modules 07 and 08. Put that fused model in a Volume, use this class shape, and you have a router in the cloud with no server to manage.

## 9. From notebook to a deployed endpoint

So far every app has been ephemeral: it exists for one `with` block. For something other people or programs call, you deploy: the app stays registered with Modal, its functions get permanent addresses, and containers start on demand and shut down when idle. Nothing bills while it sits unused.

Notebooks are good for exploring. Deployed code belongs in a Python file, partly because web endpoints and deployments need importable code, and partly because a file is what you'd commit and review. This module ships one, `endpoint_demo.py`, in the same folder as this notebook:

In [ ]:
print(Path("endpoint_demo.py").read_text())

It defines a normal function (`cube`) and a web endpoint (`square`) that calls it. Everything is on CPU, so every call costs a tiny fraction of a cent. Run these in iTerm2, from the `ML_Lab` folder, in this order:

```bash
cd ~/Developer/ML_Lab

# 1. Run the entrypoint once. Starts the app, calls cube(3) in the cloud, shuts everything down.
uv run modal run lessons/09_modal/endpoint_demo.py

# 2. Develop with a live URL. It prints a temporary link, reloads when you save the file, and stops with Ctrl+C.
uv run modal serve lessons/09_modal/endpoint_demo.py

# 3. Deploy. The URL is permanent. Modal prints it. Try it with curl, replacing the URL with yours.
uv run modal deploy lessons/09_modal/endpoint_demo.py
curl "https://<your-workspace>--ml-lab-endpoint-square.modal.run?x=5"

# 4. See what's running, and take it down.
uv run modal app list
uv run modal app stop ml-lab-endpoint     # asks you to confirm (add --yes to skip the question)
```

What to notice: `modal run` is a one-off job, `modal serve` is for development, and `modal deploy` is for production. A deployed app with no traffic costs nothing, and the first request after a quiet period pays the cold start from section 8. Always finish with `modal app list`, so you know what's left running. A deployed app keeps its URL until you stop it.

Calling a deployed class from other Python code (your FastAPI service from module 08, for example) looks like `modal.Cls.from_name("ml-lab-llm", "QwenServer")().generate.remote(...)`, and it needs your Modal token. A web endpoint like `square` can be called by anyone who has its URL, with no login. For anything public, look into `requires_proxy_auth=True` on `modal.fastapi_endpoint` before you expose a GPU to the internet.

## 10. What it costs, and how to stay in control

Per-second billing makes the arithmetic simple: dollars = rate per hour / 3600 x seconds the container ran. Two numbers decide almost every bill: how long the GPU was on, and which GPU it was. Use the `gpu_dollars` helper from section 2 to answer your own questions:

In [ ]:
# One-off jobs: how much would a run cost on each GPU?
job_seconds = 600            # change this: a 10 minute job
print(f"A {job_seconds / 60:.0f} minute job:")
for gpu in PRICE_PER_HOUR:
    print(f"  {gpu:10} ${gpu_dollars(gpu, job_seconds):.3f}")

# Serving: scale-to-zero versus keeping one container warm all month.
HOURS_PER_MONTH = 24 * 30
print("\nOne container kept warm around the clock (min_containers=1), per month:")
for gpu in ("T4", "L4", "A10", "H100"):
    print(f"  {gpu:6} ${PRICE_PER_HOUR[gpu] * HOURS_PER_MONTH:7.0f}")

# The same T4 used only while requests run: 1,000 requests a day at 0.5 s of GPU time each.
requests_per_day, seconds_each = 1000, 0.5
monthly = gpu_dollars("T4", requests_per_day * seconds_each * 30)
print(f"\nT4 on scale-to-zero, {requests_per_day:,} requests/day x {seconds_each} s: ${monthly:.2f} per month (plus cold-start time)")

The last two lines are section 1's trade-off in dollars. Always on, a T4 costs hundreds of dollars a month whether anyone uses it or not. Billed only while requests run, the same traffic costs a few dollars. The gap closes as traffic grows and the warm GPU stays busy. Past that point, a reserved or always-on GPU is cheaper per request. Exercise 5 finds the break-even for your own numbers.

Guardrails worth setting up before you rely on any of this:

1. Set a workspace spending limit in your Modal dashboard, and check usage with `uv run modal billing summary`.
2. Put `timeout=` on every function (the cells here do), so a stuck job stops by itself.
3. Put `max_containers=` on any GPU function you `map`, or that a web endpoint can trigger.
4. Treat `min_containers` as a decision. It means paying for idle GPU time.
5. After a session, run `uv run modal app list` and stop anything you don't recognize with `uv run modal app stop <name>`.
6. Delete Volumes you're done with, because they bill for storage (about 9 cents per GiB per month, so a 1 GB model is cheap, but 200 GB of checkpoints isn't): `uv run modal volume delete ml-lab-models`.

When should you use Modal?

| Situation | A good fit |
|---|---|
| Learning, small models, quick experiments | Your M5. It's free per run |
| A burst of GPU work: a sweep, a one-time fine-tune, scoring a dataset | Modal. Parallel, per-second, nothing left running |
| A model that needs more than 24 GB, such as a 7B in bf16 or a 32B | Modal or a rented box |
| A prototype API with unpredictable traffic | Modal with scale-to-zero |
| A GPU that would be busy around the clock | A reserved GPU or rented box. Modal's per-second rate is higher than a box you keep full |
| No custom model at all | A hosted API. No machine to think about |

Modal also sells hosted inference by the token for some open models (`uv run modal endpoint --help`), which sits between the last two rows. This module doesn't cover it.

## Wrap-up

Why this matters: most of what you build later needs more hardware than a laptop for some of its life. A founder who can say "this job costs $4 and finishes in 10 minutes, and that endpoint costs $12 a month at today's traffic and scales when it grows" can plan a product around real numbers, not guesses. Modal turns GPU capacity into a function call and a per-second line item, and the habits in this module (timeouts, scale limits, cleanup, cost arithmetic before a run) carry over to every cloud provider.

You're done when you can explain what an App, an Image, a Function, a Volume and a Secret are, run a function on a GPU from this notebook and read its cost off the rate and the seconds, say why the second call in section 8 was faster than the first, and shut down and clean up everything you started.

Going further:

- Run module 07's LoRA fine-tune on a rented GPU. Put the data and the base model in a Volume, and wrap a fine-tuning script in a function with `gpu="A100-80GB"`. On NVIDIA you'd use PyTorch tooling such as Hugging Face `peft` and `trl` instead of MLX. Save the adapter back to a Volume. This is the standard way to fine-tune models that don't fit on your Mac.
- `modal.Sandbox` runs untrusted or generated code in an isolated container, which matters once an LLM is writing code.
- `schedule=modal.Cron("0 9 * * *")` on a function runs it on a schedule, such as a nightly job.
- Modal's own guides: modal.com/docs/guide (GPUs, Volumes, web endpoints, cold-start tuning).

---
# Exercises

**Exercise 1: Same code, three GPUs.** Register the section 5 benchmark on a T4, an L4, and an A10 (a loop with `app.function(gpu=...)` and a different `name=` for each works). Run float16 at n = 4096 on all three. Make a table of milliseconds, TFLOPS, dollars per hour, and TFLOPS per dollar-hour. Which GPU gives the most compute per dollar, and is it the fastest one? (A few cents.)

**Exercise 2: Parallel prompts.** Using the warm `QwenServer` from section 8, send 12 different prompts with `.map()` and compare the time against sending them one at a time with `.remote()`. Both runs should happen inside one `app.run()` so the container stays warm. Is `map` faster here? Count the "model loaded" lines in the output, and explain what you see. When would `map` win? (A cent or two.)

**Exercise 3: Bigger GPUs.** Switch `RUN_BIGGER_GPUS` to `True` and run float16 at n = 8192 on an A100-40GB and an H100. How much faster than the T4 is each, and how does the cost of the run compare? Look at the wall clock too. What does that tell you about when a faster GPU is worth it? (A few cents.)

**Exercise 4: Clean up.** In iTerm2, list your apps and Volumes with `modal app list` and `modal volume list`, look inside `ml-lab-models` with `modal volume ls`, and delete the Volume when you're done with the module. Then confirm the list is clean. Explain to yourself what would have kept billing if you hadn't.

**Exercise 5: Break-even.** Using the warm latency from section 8, work out how many requests a day make an always-on T4 (`min_containers=1`) cheaper than scale-to-zero. Then say what else you'd weigh that the dollar figure ignores (hint: what does a user see on the first request after a quiet hour?).

---
# Solutions

In [ ]:
# Exercise 1
app_cmp = modal.App("ml-lab-gpu-compare", image=torch_image)
bench_on = {gpu: app_cmp.function(gpu=gpu, timeout=300, name=f"bench_{gpu.lower()}")(matmul_benchmark_impl)
            for gpu in ("T4", "L4", "A10")}

results = on_modal(app_cmp, lambda: {gpu: fn.remote(4096, "float16") for gpu, fn in bench_on.items()})
if results:
    print(f"{'GPU':5} {'ms':>7} {'TFLOPS':>7} {'$/hour':>7} {'TFLOPS per $/h':>15}")
    for gpu, r in results.items():
        print(f"{gpu:5} {r['ms']:7.2f} {r['tflops']:7.1f} {PRICE_PER_HOUR[gpu]:7.2f} {r['tflops'] / PRICE_PER_HOUR[gpu]:15.1f}")
    # The cheapest GPU per hour is often not the best value per unit of compute, and the fastest is rarely the best value.

In [ ]:
# Exercise 2
prompts = [f"Give one fun fact about the number {i}, in one sentence." for i in range(1, 13)]


def sequential_vs_map():
    server = QwenServer()
    server.generate.remote("warm up")                       # make sure the container is loaded and warm
    t0 = time.perf_counter()
    one_by_one = [server.generate.remote(p) for p in prompts]
    t1 = time.perf_counter()
    in_parallel = list(server.generate.map(prompts))
    t2 = time.perf_counter()
    return one_by_one, t1 - t0, in_parallel, t2 - t1


out = on_modal(app_llm, sequential_vs_map)
if out:
    _, seq, par, par_time = out
    print(f"one by one: {seq:.1f} s | map: {par_time:.1f} s | speedup {seq / par_time:.1f}x")
    # In a test run map was no faster (13.0 s against 13.3 s). Modal answered the 12 parallel requests by starting
    # several new containers, and each one paid a container start plus a model load of a few seconds, which ate
    # the gain. map wins when each task is long compared with that startup, or when containers are already warm
    # (min_containers). Letting one container take several inputs at once (modal.concurrent) is the next step.

In [ ]:
# Exercise 3
if RUN_BIGGER_GPUS:
    app_big = modal.App("ml-lab-gpu-big", image=torch_image)
    big = {gpu: app_big.function(gpu=gpu, timeout=300, name=f"big_{gpu.lower().replace('-', '_')}")(matmul_benchmark_impl)
           for gpu in ("A100-40GB", "H100")}
    t0 = time.perf_counter()
    results = on_modal(app_big, lambda: {gpu: fn.remote(8192, "float16") for gpu, fn in big.items()})
    wall = time.perf_counter() - t0
    if results:
        for gpu, r in results.items():
            print(f"{gpu:10} {r['ms']:7.2f} ms  {r['tflops']:6.1f} TFLOPS")
        print(f"wall clock {wall:.0f} s, about ${gpu_dollars('H100', wall) + gpu_dollars('A100-40GB', wall):.2f} at most for both")
    # The matmul itself takes milliseconds on every GPU here. For a job this short, startup dominates the bill,
    # and a bigger GPU only pays off when the job runs long enough for its speed to matter.
else:
    print("RUN_BIGGER_GPUS is False. Set it to True in the setup cell to run this one.")

In [ ]:
# Exercise 4: run these in iTerm2 from ML_Lab, or here (the delete is commented out on purpose).
!uv run modal app list
!uv run modal volume list
!uv run modal volume ls ml-lab-models
# !uv run modal volume delete ml-lab-models    # permanent: removes the model files from Modal's cloud
# Anything left in `modal app list` that you don't want: uv run modal app stop <name>
# Left running, a min_containers=1 GPU class bills around the clock. A Volume bills for storage every month.

In [ ]:
# Exercise 5
warm_seconds = 0.5            # replace with the warm latency you measured in section 8
gpu = "T4"
always_on_per_day = PRICE_PER_HOUR[gpu] * 24
per_request = gpu_dollars(gpu, warm_seconds)
break_even = always_on_per_day / per_request
print(f"Always-on {gpu}: ${always_on_per_day:.2f} per day")
print(f"Scale-to-zero: ${per_request:.6f} per request (GPU time only, cold starts add more)")
print(f"Break-even: about {break_even:,.0f} requests per day, {break_even / 86400:.1f} per second on average")
# The dollar figure ignores latency. Scale-to-zero users sometimes wait through a cold start (seconds to a
# minute, depending on model size). Always-on gives steady latency, which is often the real reason to pay for it,
# and a reserved machine can beat both once traffic keeps a GPU busy most of the day.